requests for importing dataset

In [1]:
from pathlib import Path
import requests

DATA_PATH = Path("data")
PATH = DATA_PATH / "mnist"

PATH.mkdir(parents=True, exist_ok=True)

URL = "https://github.com/pytorch/tutorials/raw/main/_static/"
FILENAME = "mnist.pkl.gz"

if not (PATH / FILENAME).exists():
        content = requests.get(URL + FILENAME).content
        (PATH / FILENAME).open("wb").write(content)

Storing using pickle, useful for our project

In [2]:
import pickle
import gzip

with gzip.open((PATH / FILENAME).as_posix(), "rb") as f:
        ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding="latin-1")

Showing some examples of data from mnist dataset, reshaped into 2d form a flattened row

In [3]:
from matplotlib import pyplot
import numpy as np

pyplot.imshow(x_train[0].reshape((28, 28)), cmap="gray")
pyplot.show()
print(x_train.shape)

ValueError: Output array must be a NumPy array

<Figure size 640x480 with 1 Axes>

(50000, 784)


In [4]:
pyplot.imshow(x_train[5].reshape((28, 28)), cmap="Oranges")
pyplot.show()
print(x_train.shape)

ValueError: Output array must be a NumPy array

<Figure size 640x480 with 1 Axes>

(50000, 784)


In [5]:
pyplot.imshow(x_train[100].reshape((28, 28)), cmap="Oranges")
pyplot.show()
print(x_train.shape)

ValueError: Output array must be a NumPy array

<Figure size 640x480 with 1 Axes>

(50000, 784)


In [6]:
import torch

x_train, y_train, x_valid, y_valid = map(
    torch.tensor, (x_train, y_train, x_valid, y_valid)
)
n, c = x_train.shape
print(x_train, y_train)
print(x_train.shape)
print(y_train.min(), y_train.max())

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]) tensor([5, 0, 4,  ..., 8, 4, 8])
torch.Size([50000, 784])
tensor(0) tensor(9)


Implementing NN

In [7]:
import math

weights = torch.randn(784, 10) / math.sqrt(784)
weights.requires_grad_()
bias = torch.zeros(10, requires_grad=True)

In [8]:
def log_softmax(x):
    return x - x.exp().sum(-1).log().unsqueeze(-1)

def model(xb):
    return log_softmax(xb @ weights + bias)

In [9]:
bs = 64  # batch size

xb = x_train[0:bs]  # a mini-batch from x
preds = model(xb)  # predictions
preds[0], preds.shape
print(preds[0], preds.shape)

tensor([-2.3249, -2.1641, -3.0171, -2.2793, -2.4588, -1.6150, -2.8741, -2.1250,
        -2.6195, -2.2742], grad_fn=<SelectBackward0>) torch.Size([64, 10])


preds variable will be used later for back propagation too

In [10]:
def nll(input, target):
    return -input[range(target.shape[0]), target].mean()

loss_func = nll

In [11]:
yb = y_train[0:bs]
print(loss_func(preds, yb))

tensor(2.2792, grad_fn=<NegBackward0>)


In [12]:
def accuracy(out, yb):
    preds = torch.argmax(out, dim=1)
    return (preds == yb).float().mean()
print(accuracy(preds, yb))

tensor(0.0938)


Below a training loop for logistsic regression.
set_trace() for debugging purposes

In [13]:
from IPython.core.debugger import set_trace

lr = 0.5  # learning rate
epochs = 2  # how many epochs to train for

for epoch in range(epochs):
    for i in range((n - 1) // bs + 1):
        #set_trace()
        start_i = i * bs
        end_i = start_i + bs
        xb = x_train[start_i:end_i]
        yb = y_train[start_i:end_i]
        pred = model(xb)
        loss = loss_func(pred, yb)

        loss.backward()
        with torch.no_grad():
            weights -= weights.grad * lr
            bias -= bias.grad * lr
            weights.grad.zero_()
            bias.grad.zero_()

In [14]:
print(loss_func(model(xb), yb), accuracy(model(xb), yb))

tensor(0.0832, grad_fn=<NegBackward0>) tensor(1.)


Now, already using the PyTorch nn class

In [15]:
import torch.nn.functional as F

loss_func = F.cross_entropy

def model(xb):
    return xb @ weights + bias

print(loss_func(model(xb), yb), accuracy(model(xb), yb))

tensor(0.0832, grad_fn=<NllLossBackward0>) tensor(1.)


In [16]:
from torch import nn

class Mnist_Logistic(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(784, 10) / math.sqrt(784))
        self.bias = nn.Parameter(torch.zeros(10))

    def forward(self, xb):
        return xb @ self.weights + self.bias

In [17]:
model = Mnist_Logistic()
model_convolution = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3)

In [18]:
xb_conv = xb.unsqueeze(1)
#yb_conv = yb.view(16, 782)

#print(f"Reshaped xb: {xb_conv.shape} \n Reshaped yb: {yb_conv.shape}")


In [19]:
print(loss_func(model(xb), yb))
#print(loss_func(model_convolution(xb_conv), yb_conv))

tensor(2.3531, grad_fn=<NllLossBackward0>)


In [20]:
def fit():
    for epoch in range(epochs):
        for i in range((n - 1) // bs + 1):
            start_i = i * bs
            end_i = start_i + bs
            xb = x_train[start_i:end_i]
            yb = y_train[start_i:end_i]
            pred = model(xb)
            loss = loss_func(pred, yb)

            loss.backward()
            with torch.no_grad():
                for p in model.parameters():
                    p -= p.grad * lr
                model.zero_grad()

fit()
print(loss_func(model(xb), yb))

tensor(0.0779, grad_fn=<NllLossBackward0>)


In [21]:
print(f"Shape of xb before Conv1d: {xb.shape}")

Shape of xb before Conv1d: torch.Size([16, 784])


In [22]:
'''
def fit():
    for epoch in range(epochs):
        for i in range((n - 1) // bs + 1):
            start_i = i * bs
            end_i = start_i + bs
            xb = x_train[start_i:end_i]
            yb = y_train[start_i:end_i]
            pred = model_convolution(xb)
            loss = loss_func(pred, yb)

            loss.backward()
            with torch.no_grad():
                for p in model_convolution.parameters():
                    p -= p.grad * lr
                model_convolution.zero_grad()

fit()

'''

'\ndef fit():\n    for epoch in range(epochs):\n        for i in range((n - 1) // bs + 1):\n            start_i = i * bs\n            end_i = start_i + bs\n            xb = x_train[start_i:end_i]\n            yb = y_train[start_i:end_i]\n            pred = model_convolution(xb)\n            loss = loss_func(pred, yb)\n\n            loss.backward()\n            with torch.no_grad():\n                for p in model_convolution.parameters():\n                    p -= p.grad * lr\n                model_convolution.zero_grad()\n\nfit()\n\n'

#  Basic NN with no pytorch

In [30]:
def cnn(image, kernel):
    kernel_height, kernel_width = kernel.shape
    image_height, image_width = image.shape
    
    output_height = image_height - kernel_height + 1
    output_width = image_width - kernel_width + 1
    
    output = np.zeros((output_height, output_width))
    
    for i in range(output_height):
        for j in range(output_width):
            region = image[i:i+kernel_height, j:j+kernel_width]
            output[i, j] = np.sum(region * kernel)
    
    return output

def relu(x):
    return np.maximum(0, x)

def max_pooling(feature_map, size=2, stride=2):
    output_height = feature_map.shape[0] // stride
    output_width = feature_map.shape[1] // stride
    
    pooled = np.zeros((output_height, output_width))
    
    for i in range(output_height):
        for j in range(output_width):
            region = feature_map[i*stride:i*stride+size, j*stride:j*stride+size]
            pooled[i, j] = np.max(region)
    
    return pooled

def fully_connected(flattened_input, weights, bias):
    return np.dot(flattened_input, weights) + bias


In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

In [25]:
def get_data_loader(batch_size=64):
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
    test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)
    
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader

train_loader, test_loader = get_data_loader()

In [31]:
kernel = np.array([[1, 0], [0, -1]])
bias = np.random.rand()
learning_rate = 0.01

train_loader, test_loader = get_data_loader()
for epoch in range(5):
    for images, labels in train_loader:
        image = images[0].squeeze().detach().cpu().numpy().astype(np.float64)
        
        # Forward pass
        feature_map = cnn(image, kernel)
        feature_map = relu(feature_map)
        pooled_output = max_pooling(feature_map)
        flattened = pooled_output.flatten()
        
        # Dynamically initialize weights based on flattened size
        if 'weights' not in locals():
            weights = np.random.rand(flattened.shape[0]).astype(np.float64)
        
        output = fully_connected(flattened, weights, bias)
        
        # Compute loss (mean squared error for simplicity)
        target = float(labels[0].item())
        loss = (output - target) ** 2
        
        # Backpropagation (gradient descent update)
        grad_output = 2 * (output - target)
        grad_weights = grad_output * flattened
        grad_bias = grad_output
        
        weights -= learning_rate * grad_weights
        bias -= learning_rate * grad_bias
    
    print(f"Epoch {epoch+1}, Loss: {loss}")

/var/folders/b9/9dk9hwxs0f7cmblj9b9s_9gr0000gn/T/ipykernel_55410/2884497134.py:34: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.dot(flattened_input, weights) + bias


RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.